# Dupin · Fase 4 — Modelado y publicación del bundle

Cambia el baseline (HistGradientBoosting) por el modelo final (**LightGBM**), lo
pasa por el MISMO régimen honesto de la Fase 3, y publica el **bundle** versionado
a `gs://dupin-dupin-artifacts/m-v1/`.

El modelo no es el protagonista. El entregable es: el bundle (modelo + esquema +
umbrales + metadata) y el contraste entre el punto **desplegado** (umbral val→test)
y la **envolvente honesta** (recall alcanzable al presupuesto exacto).

Umbrales: **review** = top ~1%, **block** = top ~0.1%, fijados sobre validación.

## 1. Clonar repo + dependencias

In [ ]:
from google.colab import userdata
import sys, subprocess
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
subprocess.run(["rm","-rf","/content/dupin"])
subprocess.run(["git","clone",f"https://{GITHUB_TOKEN}@github.com/alexxcode/dupin.git","/content/dupin"], check=True)
if "/content/dupin" not in sys.path: sys.path.insert(0,"/content/dupin")
subprocess.run(["git","-C","/content/dupin","log","--oneline","-1"])

In [ ]:
!pip -q install lightgbm gcsfs pyarrow joblib scikit-learn

## 2. Auth + cargar la matriz `feat-v1`

In [ ]:
from google.colab import auth
auth.authenticate_user()
import pandas as pd

PROJECT_ID  = "dupin-dupin"
BUCKET_FEAT = "dupin-dupin-features"
BUCKET_ART  = "dupin-dupin-artifacts"
REGION      = "us-central1"
FEAT_URI    = f"gs://{BUCKET_FEAT}/feat-v1/features.parquet"

matrix = pd.read_parquet(FEAT_URI, storage_options={"project": PROJECT_ID})
print("Matriz:", matrix.shape)

## 3. Entrenar LightGBM y construir el bundle

In [ ]:
from training.train import train_and_build
from training.model import make_lightgbm, DEFAULT_MODEL_CONFIG

bundle = train_and_build(
    matrix,
    make_model=lambda: make_lightgbm(DEFAULT_MODEL_CONFIG),
    review_budget=0.01,
    block_budget=0.001,
)
print("Bundle:", bundle.model_version, "·", bundle.feature_version)
print("Umbrales:", {k: round(v,4) for k,v in bundle.thresholds.items()})

## 4. El número honesto — desplegado vs envolvente

In [ ]:
tm = bundle.metadata["test_metrics"]
dr = tm["deployed_review"]; er = tm["envelope_review"]
print(f"PR-AUC test (prevalencia {tm['test_prevalence']:.4f}): {tm['pr_auc']:.4f}")
print()
print("Punto DESPLEGADO (umbral fijado en validación, aplicado al test):")
print(f"  review: recall={dr['recall']:.4f}  precision={dr['precision']:.4f}  review_rate={dr['review_rate']:.4f}")
print(f"  block : recall={tm['deployed_block']['recall']:.4f}  precision={tm['deployed_block']['precision']:.4f}  review_rate={tm['deployed_block']['review_rate']:.4f}")
print()
print("ENVOLVENTE honesta (recall alcanzable gastando 1% exacto en test):")
print(f"  recall={er['recall']:.4f}  precision={er['precision']:.4f}  review_rate={er['review_rate']:.4f}")
print()
print("Comparativa baseline Fase 3 (HistGB, desplegado): recall 20.7% @ review 0.46%")

## 5. Curva recall-vs-presupuesto (la envolvente)

In [ ]:
import matplotlib.pyplot as plt
curve = tm["recall_review_curve"]
rates = [op["review_rate"] for op in curve]
recs  = [op["recall"] for op in curve]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot([r*100 for r in rates], [r*100 for r in recs], marker="o")
ax.set_xlabel("Presupuesto de revisión (% operaciones)")
ax.set_ylabel("Recall (% fraude atrapado)")
ax.set_title("Envolvente honesta sobre el test temporal — LightGBM")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 6. Importancia de features (nativa de LightGBM)

In [ ]:
import pandas as pd
imp = pd.Series(bundle.model.feature_importances_, index=bundle.feature_names).sort_values(ascending=False)
display(imp.to_frame("importance"))

## 7. Publicar el bundle a GCS (`artifacts/m-v1/`)

In [ ]:
import os
from google.cloud import storage

LOCAL_DIR = "/content/bundle_m-v1"
bundle.save(LOCAL_DIR)
print("Bundle local:", os.listdir(LOCAL_DIR))

client = storage.Client(project=PROJECT_ID)
if not client.bucket(BUCKET_ART).exists():
    client.create_bucket(BUCKET_ART, location=REGION)
    print("Bucket creado:", BUCKET_ART)

for fname in os.listdir(LOCAL_DIR):
    blob = client.bucket(BUCKET_ART).blob(f"m-v1/{fname}")
    blob.upload_from_filename(os.path.join(LOCAL_DIR, fname))
    print("Subido:", f"gs://{BUCKET_ART}/m-v1/{fname}")

In [ ]:
# Verificación: recargar el bundle desde local valida que está completo.
from training.export import ModelBundle
reloaded = ModelBundle.load(LOCAL_DIR)
print("Bundle recargado OK ·", reloaded.model_version, "· features:", len(reloaded.feature_names))
print("Decisión de ejemplo (score=block_thr):", reloaded.decide_score(reloaded.thresholds["block"]))

---
**Fase 4 completa** cuando este notebook corre limpio. Salida:
`gs://dupin-dupin-artifacts/m-v1/` (model.joblib + manifest.json).

El serving (Fase 5) descarga ESTE bundle al arrancar, usa el mismo `features/`, y
rechaza arrancar si falta un componente. Los umbrales y el esquema viajan dentro.